In [1]:
import os
import numpy as np
import networkx as nx

from src.OrganoidMesh import OrganoidMesh
from src.nuclei_to_mesh_projection import *

### Setup paths to meshes and nuclei tables

In [2]:
# data_dir = '../NicoleData/20250929/fractal_output'

# # --- path to your cell nucleus table ---
# CELLS_CSV = "../NicoleData/features_csv/features/cell_types_class.csv"   # <-- adjust

# timepoint = "day4p5"
# zarr_name = "r0.zarr"
# well = "A06"
# round_name = "0_fused_zillum_registered"
# organoid_id = 31 #19 # example


# mesh_path = f"{data_dir}/{timepoint}/{zarr_name}/{well[0]}/{well[1:]}/{round_name}/meshes/nnorg_linked_multi_annotated_class/{organoid_id}.vtp"
# cells_df = pd.read_csv(CELLS_CSV)



data_dir = '../NicoleData/fractal_output'

# --- path to your cell nucleus table ---
CELLS_CSV = "../NicoleData/cell_types_class.csv"   # <-- adjust

timepoint = "day4p5"
zarr_name = "r0.zarr"
well = "A06"
round_name = "0_fused_zillum_registered"
organoid_id = 19 #19 # example


mesh_path = f"{data_dir}/{timepoint}/{zarr_name}/{well[0]}/{well[1:]}/{round_name}/meshes/nnorg_linked_multi_annotated_class/{organoid_id}.vtp"
cells_df = pd.read_csv(CELLS_CSV)



### Project binary markers to mesh

In [3]:
# Extract nuclei data matching the mesh label
organoid_id = os.path.splitext(os.path.basename(mesh_path))[0]
label_uid = f"{timepoint}_{well}_{organoid_id}"

nuclei_df_org = cells_df[cells_df["label_uid"] == label_uid].copy()

# Load membrane mesh as an OrganoidMesh
mesh = OrganoidMesh(mesh_path)

# Extract per-cell attributes from sub-table
nuclei_xyz, markers_bin = extract_cell_attributes(nuclei_df_org)

# Rescale coordinates and filter expression
_, nuclei_xyz = center_and_rescale_mesh(mesh, nuclei_xyz)
markers_bin = filter_lgr5_coexpression(markers_bin) 

# Project nuclei -> mesh vertices (using geometry from the mesh object)
proj_vertex_ids, proj_points = project_nuclei_to_mesh(
    nuclei_xyz,
    mesh,
    resolve_duplicates=True, # if multiple nuclei are projected to the same vertex, shift them slightly
)

# Compute Laplace-Beltrami eigen-decomposition for geodesics 
mesh._eig_decomp()

# Geodesic distances + Voronoi assignment
dist_mat, vertex_owner = compute_geodesic_voronoi(mesh, proj_vertex_ids)

# Assign cell quantities to mesh
marker_fields = assign_cell_quantity_to_vertices(vertex_owner, markers_bin)

print(markers_bin.shape)
print(marker_fields.shape)

Duplicate projections: none.


heat sources: 100%|██████████| 687/687 [00:10<00:00, 64.27it/s]


(687, 10)
(18001, 10)


In [ ]:
from src.nuclei_to_mesh_projection_light import *

mesh_v = mesh.v
mesh_f = mesh.f

proj_vertex_ids, proj_points = project_nuclei_to_mesh(
    nuclei_xyz,
    mesh_v,
    mesh_f,
    resolve_duplicates=True, # if multiple nuclei are projected to the same vertex, shift them slightly
    max_dist=0.1,
)


nbrs, wts = build_edge_adjacency(mesh_v, mesh_f)
owner, dist = voronoi_on_mesh_multisource(nbrs, wts, proj_vertex_ids)


uu = np.unique(owner)

print(len(uu))
print(uu)

Duplicate projections: none.
628
[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  39  40  42  43  44  45  46  47  48  49  50  51  52  53  54  55
  56  57  58  59  60  61  62  63  64  65  67  69  70  71  72  73  74  76
  77  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94
  95  96  97  99 102 103 104 105 106 108 109 110 111 112 113 114 115 116
 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134
 135 136 137 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153
 154 155 156 157 159 160 162 163 164 165 166 167 168 169 170 171 172 173
 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191
 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209
 210 211 212 213 214 215 216 217 218 219 220 222 223 224 225 226 228 229
 231 232 233 234 235 236 238 239 240 241 242 243 244 245 246 247 248 249
 250 251 252 253 2

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'mesh3d'}] * 2],
    horizontal_spacing=0.02,   # reduce gap between panels
)

fig.add_trace(
    go.Mesh3d(
        x=mesh.v[:, 0], y=mesh.v[:, 1], z=mesh.v[:, 2],
        i=mesh.f[:, 0], j=mesh.f[:, 1], k=mesh.f[:, 2],
        intensity=vertex_owner,
        colorscale='Viridis',
        showscale=False,        # <- big whitespace culprit
    ),
    row=1, col=1
)

fig.add_trace(
    go.Mesh3d(
        x=mesh.v[:, 0], y=mesh.v[:, 1], z=mesh.v[:, 2],
        i=mesh.f[:, 0], j=mesh.f[:, 1], k=mesh.f[:, 2],
        intensity=owner,
        colorscale='Viridis',
        showscale=False,
    ),
    row=1, col=2
)

# Tight scene settings
scene_common = dict(
    aspectmode='data',
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    zaxis=dict(visible=False),
)

fig.update_layout(
    scene=scene_common,
    scene2=scene_common,

    # Make figure physically larger
    width=1400,
    height=650,

    # Remove outer margins
    margin=dict(l=0, r=0, t=30, b=0),
)

fig.show()


In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

marker_names = ['LGR5', 'Chromogranin A', 'Cyclin D', 'Mucin 2', 'AldoB',
                'Glucagon', 'Cyclin A', 'Agr2', 'Serotonin', 'Lysozyme']

# --- Markers to plot ---
marker_list = ["Serotonin", "Lysozyme", "LGR5"]

# --- Get their indices ---
marker_indices = [marker_names.index(name) for name in marker_list]

# --- Create subplot figure ---
fig = make_subplots(
    rows=1, cols=len(marker_list),
    specs=[[{'type': 'mesh3d'}] * len(marker_list)],
    subplot_titles=marker_list
)

# --- Add each marker as a Mesh3d trace ---
for i, (name, idx) in enumerate(zip(marker_list, marker_indices), start=1):
    fig.add_trace(
        go.Mesh3d(
            x=mesh.v[:, 0], y=mesh.v[:, 1], z=mesh.v[:, 2],
            i=mesh.f[:, 0], j=mesh.f[:, 1], k=mesh.f[:, 2],
            intensity=marker_fields[:, idx],
            colorscale='Viridis',
            showscale=True,
            name=name
        ),
        row=1, col=i
    )

# --- Adjust layout ---
for j in range(len(marker_list)):
    fig.update_layout(**{f"scene{'' if j == 0 else j+1}": dict(aspectmode='data')})

fig.update_layout(
    title_text="Marker intensity maps: Serotonin, Lysozyme, LGR5",
    showlegend=False
)

fig.show()

In [11]:

# Build arrays for the connection segments
line_x = []
line_y = []
line_z = []

for p_nuc, p_proj in zip(nuclei_xyz, proj_points):
    line_x += [p_nuc[0], p_proj[0], None]
    line_y += [p_nuc[1], p_proj[1], None]
    line_z += [p_nuc[2], p_proj[2], None]

fig = go.Figure(data=[
    # Connection lines: nucleus -> projected point
    go.Scatter3d(
        x=line_x,
        y=line_y,
        z=line_z,
        mode="lines",
        line=dict(width=2, color="rgba(0,0,0,0.5)"),  # semi-transparent gray
        name="Projection lines"
    ),

    # Projected points on the surface (red)
    go.Scatter3d(
        x=proj_points[:, 0],
        y=proj_points[:, 1],
        z=proj_points[:, 2],
        mode='markers',
        marker=dict(size=5, color='red', symbol='circle'),
        name="Projected nuclei on surface"
    ),

    # True nucleus centroids in volume (blue)
    go.Scatter3d(
        x=nuclei_xyz[:, 0],
        y=nuclei_xyz[:, 1],
        z=nuclei_xyz[:, 2],
        mode='markers',
        marker=dict(size=5, color='blue', symbol='circle'),
        name="Nucleus centroids"
    )
])

fig.update_layout(
    scene=dict(aspectmode='data'),
    title="Projected nuclei on surface (red) vs cell centroids (blue) with projection lines"
)

fig.show()
